1. The Data Loader

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import fetch_openml

print("🚀 Fetching reliable Adult Census Income dataset via OpenML...")
# Using OpenML ensures the dataset is always available and perfectly formatted
adult = fetch_openml(name='adult', version=2, as_frame=True, parser='auto')
df = adult.frame

# Standardize the target column name for our Audit Pipeline
df.rename(columns={'class': 'income'}, inplace=True)

# Create our binary target: 1 if high income (>50K), 0 if low income (<=50K)
df['target'] = df['income'].apply(lambda x: 1 if x == '>50K' else 0)

print(f"✅ Loaded {len(df)} demographic records.")
print("🔍 The Gatekeeper is ready to audit the 'sex' attribute.")
display(df[['sex', 'target']].head())

🚀 Fetching reliable Adult Census Income dataset via OpenML...
✅ Loaded 48842 demographic records.
🔍 The Gatekeeper is ready to audit the 'sex' attribute.


,sex,target
0,Male,0
1,Male,0
2,Male,1
3,Male,1
4,Female,0


2. The Baseline Model & The Fairness Audit

In [4]:
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

print("🧠 Training Baseline Model (The potentially biased algorithm)...")

# 1. Quick Data Prep: Encode categorical text into numbers for the model
df_encoded = df.copy()

# Explicitly map our protected attribute so we know the exact mathematical breakdown
# 1 = Privileged (Male), 0 = Unprivileged (Female)
df_encoded['sex'] = df_encoded['sex'].apply(lambda x: 1 if x == 'Male' else 0)

# Encode all other categorical columns
categorical_cols = df_encoded.select_dtypes(include=['object', 'category']).columns
for col in categorical_cols:
    if col != 'sex' and col != 'income':
        df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col].astype(str))

# 2. Isolate features and target
X = df_encoded.drop(['income', 'target'], axis=1)
y = df_encoded['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train a standard, unconstrained model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"✅ Baseline Accuracy: {accuracy_score(y_test, y_pred):.2%}")

# --- THE AUDIT GATEKEEPER ---
print("\n🔎 Initiating Fairness Audit on Protected Attribute: 'sex'")

# Re-attach predictions to our test features so we can slice them by demographic
audit_df = X_test.copy()
audit_df['prediction'] = y_pred

# Calculate the rate at which the model predicts ">50K" for each group
male_approval_rate = audit_df[audit_df['sex'] == 1]['prediction'].mean()
female_approval_rate = audit_df[audit_df['sex'] == 0]['prediction'].mean()

print(f"👔 Male High-Income Prediction Rate: {male_approval_rate:.2%}")
print(f"👗 Female High-Income Prediction Rate: {female_approval_rate:.2%}")

# Calculate Disparate Impact
disparate_impact = female_approval_rate / male_approval_rate

print(f"\n⚖️ Disparate Impact Ratio: {disparate_impact:.3f}")
if disparate_impact < 0.8:
    print("🚨 FAIL: Model deployment blocked. Severe bias detected (Ratio < 0.8).")
else:
    print("✅ PASS: Model meets standard fairness thresholds.")

🧠 Training Baseline Model (The potentially biased algorithm)...
✅ Baseline Accuracy: 86.25%

🔎 Initiating Fairness Audit on Protected Attribute: 'sex'
👔 Male High-Income Prediction Rate: 26.13%
👗 Female High-Income Prediction Rate: 8.04%

⚖️ Disparate Impact Ratio: 0.308
🚨 FAIL: Model deployment blocked. Severe bias detected (Ratio < 0.8).


Step 3: Auto-Tuning Mitigation Strategy

In [6]:
import numpy as np
from sklearn.metrics import accuracy_score

print("⚙️ Initiating Automated Fairness Tuning (Threshold Search)...")

# 1. Intercept the raw probabilities
y_probs = model.predict_proba(X_test)[:, 1]
mitigation_df = X_test.copy()
mitigation_df['true_target'] = y_test
mitigation_df['raw_probability'] = y_probs

# 2. Lock the privileged threshold
threshold_privileged = 0.50
optimal_unprivileged = None

# 3. The Auto-Tuner: Search for the exact threshold that satisfies the Gatekeeper
print("🔍 Scanning for optimal fairness threshold...")
for t in np.arange(0.50, 0.05, -0.01):
    def apply_dynamic_threshold(row):
        if row['sex'] == 1:
            return 1 if row['raw_probability'] >= threshold_privileged else 0
        else:
            return 1 if row['raw_probability'] >= t else 0

    mitigation_df['test_prediction'] = mitigation_df.apply(apply_dynamic_threshold, axis=1)

    male_rate = mitigation_df[mitigation_df['sex'] == 1]['test_prediction'].mean()
    female_rate = mitigation_df[mitigation_df['sex'] == 0]['test_prediction'].mean()

    # Prevent division by zero
    if male_rate == 0:
        continue

    di_ratio = female_rate / male_rate

    # The moment we hit legal compliance, lock it in and stop searching
    if di_ratio >= 0.8:
        optimal_unprivileged = t
        break

# --- FINAL AUDIT REPORT ---
print("\n" + "="*40)
print("🛡️ FINAL GATEKEEPER COMPLIANCE REPORT")
print("="*40)

if optimal_unprivileged:
    print(f"🎯 Tuning Complete: Unprivileged Threshold locked at {optimal_unprivileged:.2f}")
    print(f"👔 Male High-Income Rate: {male_rate:.2%}")
    print(f"👗 Female High-Income Rate: {female_rate:.2%}")
    print(f"⚖️ Final Disparate Impact Ratio: {di_ratio:.3f}")

    new_accuracy = accuracy_score(mitigation_df['true_target'], mitigation_df['test_prediction'])
    print(f"📉 Final Pipeline Accuracy: {new_accuracy:.2%} (The mathematical cost of fairness)")
    print("\n✅ PASS: Mitigation successful. Gatekeeper unlocked. Model approved for deployment.")
else:
    print("🚨 CRITICAL FAIL: Algorithm could not find a compliant threshold.")

⚙️ Initiating Automated Fairness Tuning (Threshold Search)...
🔍 Scanning for optimal fairness threshold...

🛡️ FINAL GATEKEEPER COMPLIANCE REPORT
🎯 Tuning Complete: Unprivileged Threshold locked at 0.14
👔 Male High-Income Rate: 26.69%
👗 Female High-Income Rate: 21.92%
⚖️ Final Disparate Impact Ratio: 0.821
📉 Final Pipeline Accuracy: 83.50% (The mathematical cost of fairness)

✅ PASS: Mitigation successful. Gatekeeper unlocked. Model approved for deployment.
